Estimating OLS and Poisson regression models for ICE enforcement intensity
at the state level. Running a 999-iteration permutation placebo test to
evaluate whether the AAI-enforcement association exceeds chance.

Model specification
-------------------
Main OLS (HC3 robust standard errors, N=51 states):

    enforcement_rate = β₀
                     + β₁·AAI
                     + β₂·(AAI × foreign_born_share)
                     + β₃·foreign_born_share
                     + β₄·poverty_rate
                     + β₅·uninsured_share_proxy
                     + β₆·state_287g
                     + β₇·border_state
                     + β₈·sanctuary_policy
                     + ε

HC3 (MacKinnon-White) robust SEs are used to account for heteroskedasticity
in small samples (N=51). The interaction term tests whether the AAI effect
is amplified in states with larger immigrant populations.

Poisson models estimated as supplementary specification with log(population)
offset. Results consistent with OLS for policy coefficients.

Permutation test
----------------
999 random permutations of AAI values across states, preserving all other
variables. Empirical p-value = share of null coefficients with absolute value ≥ observed coefficient.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parents[2] / "shared"))

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from config import DATA_PROCESSED, RESULTS_P1

RNG = np.random.default_rng(42)

CONTROLS = (
    "foreign_born_share + poverty_rate + uninsured_share_proxy "
    "+ state_287g + border_state + sanctuary_policy"
)


def load() -> pd.DataFrame:
    df = pd.read_csv(DATA_PROCESSED / "state_analysis_table.csv",
                     dtype={"state_fips": str})
    df = df[df["population"] > 0].copy()
    df["log_population"] = np.log(df["population"])
    numeric_cols = [
        "aai", "aai_x_foreign_born", "arrests_per_100k", "detainers_per_100k",
        "foreign_born_share", "poverty_rate", "uninsured_share_proxy",
        "state_287g", "border_state", "sanctuary_policy",
    ]
    for col in numeric_cols:
        df[col] = pd.to_numeric(df.get(col, 0), errors="coerce").fillna(0)
    return df


def coef_table(model, model_name: str, outcome: str) -> pd.DataFrame:
    return pd.DataFrame({
        "model":      model_name,
        "outcome":    outcome,
        "term":       model.params.index,
        "coef":       model.params.values,
        "std_error":  model.bse.values,
        "p_value":    model.pvalues.values,
        "ci_low":     model.conf_int()[0].values,
        "ci_high":    model.conf_int()[1].values,
        "irr":        np.exp(model.params.values),
    })


def run_ols(df: pd.DataFrame, outcome: str, interaction: bool = True) -> object:
    terms = "aai + aai_x_foreign_born" if interaction else "aai"
    formula = f"{outcome} ~ {terms} + {CONTROLS}"
    return smf.ols(formula, data=df).fit(cov_type="HC3")


def run_poisson(df: pd.DataFrame, outcome_count: str) -> object:
    formula = f"{outcome_count} ~ aai + aai_x_foreign_born + {CONTROLS}"
    return smf.glm(
        formula, data=df,
        family=sm.families.Poisson(),
        offset=df["log_population"],
    ).fit(cov_type="HC1")


def permutation_test(df: pd.DataFrame, outcome: str,
                     n_perm: int = 999) -> dict:


State-level permutation placebo test.

Shuffles AAI values across states 999 times and computes the empirical p-value for the observed AAI coefficient.
    


In [ ]:
    formula = f"{outcome} ~ aai + aai_x_foreign_born + {CONTROLS}"
    obs_coef = smf.ols(formula, data=df).fit(cov_type="HC3").params["aai"]

    null_dist = []
    tmp = df.copy()
    for _ in range(n_perm):
        tmp["aai"] = RNG.permutation(df["aai"].values)
        tmp["aai_x_foreign_born"] = tmp["aai"] * tmp["foreign_born_share"]
        try:
            coef = smf.ols(formula, data=tmp).fit().params["aai"]
            null_dist.append(coef)
        except Exception:
            continue

    null_arr = np.array(null_dist)
    emp_p = (np.sum(np.abs(null_arr) >= abs(obs_coef)) + 1) / (len(null_arr) + 1)

    pd.DataFrame({"null_coef": null_arr}).to_csv(
        RESULTS_P1 / f"permutation_null_{outcome}.csv", index=False
    )
    return {
        "outcome":      outcome,
        "observed_coef": obs_coef,
        "empirical_p":  emp_p,
        "n_perm":       len(null_arr),
        "interpretation": "null confirmed" if emp_p > 0.1 else "significant",
    }


def main():
    df = load()
    print(f"N = {len(df)} states\n")

    tables, summaries = [], []

    # OLS models
    for rate, count in [("arrests_per_100k","arrests"), ("detainers_per_100k","detainers")]:
        for interact, label in [(True,"with_interaction"), (False,"main")]:
            m = run_ols(df, rate, interact)
            tables.append(coef_table(m, f"OLS_HC3_{label}", rate))
            summaries.append(f"\n\n{'='*60}\n{rate} / OLS {label}\n{'='*60}\n{m.summary()}")
            print(f"OLS {label} — {rate}:")
            print(f"  AAI coef = {m.params['aai']:.3f}  p = {m.pvalues['aai']:.3f}")
            print(f"  287g coef = {m.params.get('state_287g', np.nan):.3f}  p = {m.pvalues.get('state_287g', np.nan):.3f}\n")

        # Poisson
        try:
            p = run_poisson(df, count)
            tables.append(coef_table(p, "Poisson_HC1", count))
            summaries.append(f"\n\n{'='*60}\n{count} / Poisson HC1\n{'='*60}\n{p.summary()}")
        except Exception as e:
            print(f"Poisson failed for {count}: {e}")

    pd.concat(tables, ignore_index=True).to_csv(
        RESULTS_P1 / "model_coefficients_all.csv", index=False
    )
    (RESULTS_P1 / "model_summary.txt").write_text("".join(str(s) for s in summaries))
    print("Model results saved.\n")

    # Permutation tests
    print("Running permutation tests (999 iterations each)...")
    placebo = pd.DataFrame([
        permutation_test(df, "arrests_per_100k"),
        permutation_test(df, "detainers_per_100k"),
    ])
    placebo.to_csv(RESULTS_P1 / "permutation_summary.csv", index=False)
    print("\nPermutation results:")
    print(placebo.to_string(index=False))

    # Descriptive comparisons
    df["high_aai"] = (df["aai"] >= df["aai"].median()).astype(int)
    comp_aai = df.groupby("high_aai").agg(
        n=("state_fips","count"),
        mean_arrests=("arrests_per_100k","mean"),
        mean_detainers=("detainers_per_100k","mean"),
        mean_aai=("aai","mean"),
        mean_fqhc=("fqhc_sites_per_100k","mean"),
        mean_foreign_born=("foreign_born_share","mean"),
        pct_287g=("state_287g","mean"),
        pct_sanctuary=("sanctuary_policy","mean"),
    ).reset_index()
    comp_aai.to_csv(RESULTS_P1 / "high_low_aai_comparison.csv", index=False)

    comp_287 = df.groupby("state_287g").agg(
        n=("state_fips","count"),
        mean_arrests=("arrests_per_100k","mean"),
        median_arrests=("arrests_per_100k","median"),
        mean_detainers=("detainers_per_100k","mean"),
        median_detainers=("detainers_per_100k","median"),
        mean_foreign_born=("foreign_born_share","mean"),
        mean_aai=("aai","mean"),
        pct_sanctuary=("sanctuary_policy","mean"),
    ).reset_index()
    comp_287.to_csv(RESULTS_P1 / "287g_group_comparison.csv", index=False)

    print("\n287(g) group comparison:")
    print(comp_287.to_string(index=False))
    print("\nAll results saved to results/phase1/")


if __name__ == "__main__":
    main()
